# `financialtools` — LangChain Pipeline Guide

How `chains.py`, `prompts.py`, `pydantic_models.py`, and `tools.py` wire together
into a single LCEL pipeline that returns a validated `StockRegimeAssessment`.

```
financialtools — LangChain layer
│
├── pydantic_models.py
│   └── StockRegimeAssessment          validated output schema (8 fields)
│
├── prompts.py
│   ├── build_prompt(sector_aware, include_red_flags, include_extended_metrics) → str
│   ├── system_prompt_StockRegimeAssessment           (base variant)
│   ├── system_prompt_StockRegimeAssessment_sector    (+ sector comparison)
│   ├── system_prompt_noredflags_StockRegimeAssessment (no red flags)
│   ├── system_prompt_StockRegimeAssessment_extended  (sector + 14 extended metrics)
│   └── system_prompt                                 (rich / emoji variant)
│
├── chains.py
│   └── get_stock_evaluation_report(ticker, year?, base_dir, sector_file)
│       → StockRegimeAssessment
│       reads {base_dir}/*.xlsx, maps ticker→sector via sector_file
│       → ChatPromptTemplate | ChatOpenAI | OutputFixingParser(PydanticOutputParser)
│
└── tools.py
    ├── make_tools(base_dir, sector_file) → list[@tool]   ← external consumers
    ├── TOOLS = make_tools()                              ← in-repo default
    │
    ├── list_available_tickers()          → JSON array
    ├── get_stock_metrics(ticker, year?)  → JSON {metrics, composite_scores, red_flags}
    ├── get_sector_benchmarks(sector)     → JSON {financial, valuation}
    ├── get_red_flags(ticker)             → JSON {red_flags}
    └── get_stock_regime_report(t, yr?)  → JSON (StockRegimeAssessment.model_dump())
```

| Section | Needs API key | Needs data files |
|---|---|---|
| 1 — Setup | no | no |
| 2 — StockRegimeAssessment | no | no |
| 3 — build_prompt() variants | no | no |
| 4 — ChatPromptTemplate (offline) | no | no |
| 5 — PydanticOutputParser (offline) | no | no |
| 6 — LCEL chain with mock LLM | no | no |
| 7 — LIVE: get_stock_evaluation_report | **yes** | **yes** |
| 8 — Agent tools layer | no | no |
| 9 — Error handling reference | no | no |
| 10 — End-to-end pattern | **yes** | **yes** |

## Sections
1. [Setup](#1--setup)
2. [StockRegimeAssessment — the output schema](#2--stockregimeassessment--the-output-schema)
3. [build_prompt() — five prompt variants](#3--build_prompt--five-prompt-variants)
4. [ChatPromptTemplate.from_messages() — offline](#4--chatprompttemplatefrom_messages--offline)
5. [PydanticOutputParser + OutputFixingParser — offline](#5--pydanticoutputparser--outputfixingparser--offline)
6. [LCEL chain composition — offline mock](#6--lcel-chain-composition--offline-mock)
7. [LIVE: get_stock_evaluation_report()](#7--live-get_stock_evaluation_report)
8. [Agent tools layer (tools.py)](#8--agent-tools-layer)
9. [Error handling reference](#9--error-handling-reference)
10. [End-to-end pattern](#10--end-to-end-pattern)

## 1 — Setup

In [ ]:
import json
import logging
import os

from dotenv import load_dotenv

load_dotenv()

# LangChain imports
from langchain_core.output_parsers import PydanticOutputParser
from langchain.output_parsers import OutputFixingParser
from langchain_core.prompts import ChatPromptTemplate

# financialtools imports
from financialtools.pydantic_models import StockRegimeAssessment
from financialtools.prompts import (
    build_prompt,
    system_prompt_StockRegimeAssessment,
    system_prompt_StockRegimeAssessment_sector,
    system_prompt_noredflags_StockRegimeAssessment,
    system_prompt_StockRegimeAssessment_extended,
    system_prompt,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s — %(message)s",
    force=True,
)

print("Imports OK")

## 2 — StockRegimeAssessment — the output schema

`StockRegimeAssessment` is the Pydantic v2 model that the LangChain pipeline
must return. Every field has a `description` used by `PydanticOutputParser`
to generate format instructions.

| Field | Type | Constrained | Description |
|---|---|---|---|
| `ticker` | `str` | no | The ticker under analysis |
| `regime` | `Literal["bull", "bear"]` | yes | Fundamental regime classification |
| `regime_rationale` | `str` | no | Justification drawing on metrics + composite score + red flags |
| `metrics_movement` | `str` | no | Narrative of how key metrics moved across years |
| `non_aligned_findings` | `Optional[str]` | no | Contradictory signals — `None` if none exist |
| `evaluation` | `Literal["overvalued", "undervalued", "fair"]` | yes | Valuation classification |
| `evaluation_rationale` | `str` | no | Justification for the valuation classification |
| `market_comparison` | `str` | no | Stock vs sector peers — fundamentals and valuation |

`Literal` fields are validated by Pydantic: any value outside the allowed set raises
`ValidationError`, which `OutputFixingParser` catches and retries with the LLM.

In [ ]:
# Inspect the schema and its JSON representation (offline — no API key needed)
import json

schema = StockRegimeAssessment.model_json_schema()
print(f"Title    : {schema['title']}")
print(f"Required : {schema['required']}")
print()
print("Field descriptions:")
for name, props in schema.get("properties", {}).items():
    ftype = props.get("type", props.get("anyOf", "?"))
    desc  = props.get("description", "—")[:70]
    print(f"  {name:<28} {str(ftype):<20} {desc}")

print()

# Demonstrate construction + validation (Literal field enforced)
valid = StockRegimeAssessment(
    ticker="AAPL",
    regime="bull",
    regime_rationale="Strong FCF growth and expanding margins across 4 years.",
    metrics_movement="GrossMargin steady, FCFToRevenue improving, DebtToEquity declining.",
    non_aligned_findings=None,
    evaluation="fair",
    evaluation_rationale="P/E in line with sector median; FCFYield slightly above peers.",
    market_comparison="AAPL outperforms sector on FCF metrics; in line on leverage.",
)
print("Constructed OK:")
for k, v in valid.model_dump().items():
    print(f"  {k:<28} {str(v)[:80]}")

## 3 — `build_prompt()` — five prompt variants

`prompts.py` defines five system-prompt strings built from shared metric-definition
blocks. Editing a block (e.g. renaming a metric) propagates automatically to every
variant that includes it.

```python
build_prompt(
    sector_aware: bool = False,
    include_red_flags: bool = True,
    include_extended_metrics: bool = False,
) -> str
```

| Constant exported | `sector_aware` | `include_red_flags` | `include_extended_metrics` | Use case |
|---|---|---|---|---|
| `system_prompt_StockRegimeAssessment` | `False` | `True` | `False` | Default — used by `chains.py` |
| `system_prompt_StockRegimeAssessment_sector` | `True` | `True` | `False` | Adds sector-comparison instruction |
| `system_prompt_noredflags_StockRegimeAssessment` | `False` | `False` | `False` | Omits red-flag data description |
| `system_prompt_StockRegimeAssessment_extended` | `True` | `True` | `True` | Sector-aware + 14 unscored extended metrics |
| `system_prompt` | N/A | N/A | N/A | Rich variant with emoji headers — kept as a literal |

**`include_extended_metrics=True`** appends `_EXTENDED_METRICS_BLOCK` to the prompt,
which describes 14 unscored diagnostic indicators (working capital chain, growth rates,
red-flag ratios). Use this variant when `extended_metrics` is present in the `evaluate()`
result sent to the LLM — it teaches the model what those columns mean and how to interpret them.

**Known limitation (documented TODO in `chains.py`):** `system_prompt_StockRegimeAssessment`
has no `{format_instructions}` placeholder, so the `.format(format_instructions=...)` call
in the chain is a no-op. Format instructions are never delivered to the LLM.
`OutputFixingParser` is therefore the sole recovery path for malformed output.
Fix: add `{format_instructions}` to `build_prompt()` — but validate LLM behaviour before
shipping to avoid regressions on existing output format.

In [ ]:
# Compare all five variants (offline — no API key needed)
variants = {
    "base (no sector, with red flags)" : build_prompt(sector_aware=False, include_red_flags=True),
    "sector-aware"                     : build_prompt(sector_aware=True,  include_red_flags=True),
    "no red flags"                     : build_prompt(sector_aware=False, include_red_flags=False),
    "extended (sector + ext metrics)"  : build_prompt(sector_aware=True,  include_red_flags=True,
                                                      include_extended_metrics=True),
    "rich / system_prompt"             : system_prompt,
}

for label, text in variants.items():
    print(f"{label:<42} {len(text):>5} chars")

# Lines that sector_aware=True adds over the base
base   = build_prompt(sector_aware=False, include_red_flags=True)
sector = build_prompt(sector_aware=True,  include_red_flags=True)
added  = [l for l in sector.splitlines() if l and l not in base.splitlines()]
print(f"\nLines added by sector_aware=True ({len(added)} lines):")
for l in added:
    print(f"  {l}")

# Confirm extended block is gated correctly
from financialtools.prompts import _EXTENDED_METRICS_BLOCK
ext_prompt  = build_prompt(include_extended_metrics=True)
base_prompt = build_prompt(include_extended_metrics=False)
assert "Cash Conversion Cycle" in ext_prompt
assert "Cash Conversion Cycle" not in base_prompt
print("\nExtended block gating: OK (CCC only in extended variant)")

# Preview the first 300 chars of the base prompt
print(f"\n--- Base prompt preview ---\n{base[:300]}\n...")

## 4 — `ChatPromptTemplate.from_messages()` — offline

`chains.py` assembles a two-message prompt:

```python
prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt_filled),
    ("human",  "Metrics:\n{metrics}\nScores:\n{composite_scores}\n..."),
])
```

- The **system message** carries the analyst persona and metric definitions.
- The **human message** carries six JSON payloads inlined as template variables:
  `{metrics}`, `{eval_metrics}`, `{composite_scores}`, `{red_flags}`,
  `{market_metrics}`, `{eval_market_metrics}`.

`ChatPromptTemplate` validates that all `{variable}` placeholders are present at
invocation time. Omitting any key raises `KeyError`.

In [ ]:
# Build and inspect the template without making any API calls
_HUMAN_TEMPLATE = (
    "Metrics:\n{metrics}\n"
    "Scores:\n{composite_scores}\n"
    "Evaluation Metrics:\n{eval_metrics}\n"
    "RedFlags:\n{red_flags}\n"
    "Market metrics:{market_metrics}\n"
    "Market evaluation metrics:{eval_market_metrics}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt_StockRegimeAssessment),
    ("human",  _HUMAN_TEMPLATE),
])

# Inspect template metadata
print(f"Message types       : {[m.role if hasattr(m, 'role') else type(m).__name__ for m in prompt.messages]}")
print(f"Input variables     : {sorted(prompt.input_variables)}")
print(f"Expected var count  : {len(prompt.input_variables)}")

# Format with dummy payloads — produces a list of BaseMessages (offline)
dummy = {k: f"[{k}_data]" for k in prompt.input_variables}
messages = prompt.format_messages(**dummy)
print(f"\nFormatted messages  : {len(messages)} messages")
print(f"  [0] type          : {type(messages[0]).__name__}")
print(f"  [0] content[:80]  : {messages[0].content[:80]!r}...")
print(f"  [1] type          : {type(messages[1]).__name__}")
print(f"  [1] content[:120] : {messages[1].content[:120]!r}")

## 5 — `PydanticOutputParser` + `OutputFixingParser` — offline

### PydanticOutputParser

Derives format instructions from the Pydantic model's JSON schema and attempts to
parse the LLM's text response into the model.

```python
parser = PydanticOutputParser(pydantic_object=StockRegimeAssessment)
instructions = parser.get_format_instructions()   # embed in the system prompt
result = parser.parse(llm_text_response)           # → StockRegimeAssessment
```

### OutputFixingParser

Wraps `PydanticOutputParser` with a retry loop: if the primary parse fails (bad JSON,
wrong field types, invalid `Literal` value) it makes a **second LLM call** that asks
the model to fix the malformed output.

```python
fixing_parser = OutputFixingParser.from_llm(parser=parser, llm=llm)
```

**When to use `OutputFixingParser`:**
Use it whenever the LLM may return slightly-malformed JSON (e.g. extra text before the
JSON block, wrong enum value). Avoid using it with expensive models for high-volume
pipelines — the retry doubles the token cost for every failed parse.

**Current codebase state:** `chains.py` creates the fixing parser but the format
instructions string is never injected into the prompt (the `{format_instructions}`
placeholder is missing from the prompt template). The LLM outputs JSON only because the
system prompt asks for it by describing the expected fields — `OutputFixingParser` is
therefore the primary (not the fallback) correction mechanism.

In [ ]:
# PydanticOutputParser — inspect format instructions (offline, no API call)
base_parser = PydanticOutputParser(pydantic_object=StockRegimeAssessment)

instructions = base_parser.get_format_instructions()
print(f"Format instructions length : {len(instructions)} chars")
print()
print("--- Format instructions (first 600 chars) ---")
print(instructions[:600])
print("...")

print()

# Parse a hand-crafted valid JSON string (simulates a perfect LLM response)
_VALID_JSON = json.dumps({
    "ticker": "AAPL",
    "regime": "bull",
    "regime_rationale": "Consistently strong FCF and expanding margins.",
    "metrics_movement": "GrossMargin improved; DebtToEquity stable.",
    "non_aligned_findings": None,
    "evaluation": "fair",
    "evaluation_rationale": "P/E in line with sector peers.",
    "market_comparison": "Above sector on FCF, in line on leverage.",
})

parsed = base_parser.parse(_VALID_JSON)
print(f"Parsed type  : {type(parsed).__name__}")
print(f"regime       : {parsed.regime}")
print(f"evaluation   : {parsed.evaluation}")

# Demonstrate: invalid Literal value raises OutputParserException
from langchain_core.exceptions import OutputParserException
bad_json = _VALID_JSON.replace('"bull"', '"sideways"')
try:
    base_parser.parse(bad_json)
except OutputParserException as exc:
    print(f"\nBad Literal raises OutputParserException: {str(exc)[:120]}")

## 6 — LCEL chain composition — offline mock

LCEL (LangChain Expression Language) composes runnables with the `|` operator:

```
chain = prompt | llm | parser
```

Each `|` calls `.invoke()` on the right-hand component with the output of the left.

**Data flow:**
```
dict of 6 JSON strings
  → prompt.invoke()   → list[BaseMessage]
  → llm.invoke()      → AIMessage  (text content)
  → parser.invoke()   → StockRegimeAssessment
```

This cell uses `FakeListChatModel` from `langchain_community` to simulate the LLM
response with a pre-written valid JSON string — no API key required.

In [ ]:
from langchain_community.chat_models.fake import FakeListChatModel

# Pre-written LLM response that the fake model will return
_MOCK_RESPONSE = json.dumps({
    "ticker": "AAPL",
    "regime": "bull",
    "regime_rationale": "FCF yield strong; margins improving across 4 years.",
    "metrics_movement": "GrossMargin +3pp, ROE +2pp, DebtToEquity declining.",
    "non_aligned_findings": "EarningsYield slightly below sector average.",
    "evaluation": "fair",
    "evaluation_rationale": "P/E at sector median; P/FCF slightly below.",
    "market_comparison": "Fundamentals above sector; valuation in line with peers.",
})

# Build the chain with the fake LLM
fake_llm   = FakeListChatModel(responses=[_MOCK_RESPONSE])
chain      = prompt | fake_llm | base_parser

# Dummy input — the same 6 keys the real chain receives
dummy_input = {k: f"[stub_{k}]" for k in prompt.input_variables}

result = chain.invoke(dummy_input)

print(f"Return type  : {type(result).__name__}")
print(f"ticker       : {result.ticker}")
print(f"regime       : {result.regime}")
print(f"evaluation   : {result.evaluation}")
print()
print("Full model_dump():")
for k, v in result.model_dump().items():
    print(f"  {k:<28} {str(v)[:80]}")

# Demonstrate the pipe operator produces a Runnable (can also be streamed)
print(f"\nchain type   : {type(chain).__name__}")

## 7 — LIVE: `get_stock_evaluation_report()`

> **LIVE cell** — requires:
> - `OPENAI_API_KEY` in `.env` (or environment)
> - `financial_data/<TICKER>_*.xlsx` — run `python scripts/run_pipeline.py` first
> - `financial_data/metrics_by_sectors.xlsx` and `eval_metrics_by_sectors.xlsx`
> - Ticker present in the `sector_file` mapping

```python
get_stock_evaluation_report(
    ticker: str,
    year: int | None = None,
    base_dir: str = "financial_data",
    sector_file: str = "financialtools/data/sector_ticker.txt",
) -> StockRegimeAssessment
```

Calling sequence inside `chains.py`:
1. `read_financial_results(ticker, input_dir=base_dir, ...)` — loads Excel → 4 DataFrames
2. `get_sector_for_ticker(ticker, sector_file=sector_file)` — looks up sector string
3. `get_market_metrics(file_path=base_dir/..., sector)` × 2 — loads sector benchmark DataFrames
4. `dataframe_to_json(df)` × 6 — serialise all DataFrames to JSON strings
5. `ChatPromptTemplate` + `ChatOpenAI(gpt-4.1-nano, temperature=0)` + `OutputFixingParser`
6. `chain.invoke({...})` → `StockRegimeAssessment`

In [ ]:
if not os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not set — skipping live call.")
else:
    from financialtools.chains import get_stock_evaluation_report

    TICKER = "AAPL"   # change to any evaluated ticker
    YEAR   = 2023     # optional — pass None for all years

    report = get_stock_evaluation_report(TICKER, year=YEAR)

    print(f"Ticker       : {report.ticker}")
    print(f"Regime       : {report.regime}")
    print(f"Evaluation   : {report.evaluation}")
    print()
    print("Regime rationale:")
    print(f"  {report.regime_rationale}")
    print()
    print("Metrics movement:")
    print(f"  {report.metrics_movement}")
    print()
    if report.non_aligned_findings:
        print("Non-aligned findings:")
        print(f"  {report.non_aligned_findings}")
    print()
    print("Market comparison:")
    print(f"  {report.market_comparison}")

In [ ]:
# Serialise the report — model_dump() is the Pydantic v2 method
# (tools.py uses this to return JSON to the agent)
if not os.environ.get("OPENAI_API_KEY"):
    print("OPENAI_API_KEY not set — showing schema of model_dump() keys instead.")
    print(list(StockRegimeAssessment.model_fields.keys()))
else:
    report_dict = report.model_dump()
    print("model_dump() keys:", list(report_dict.keys()))
    print()
    # Verify round-trip: dict → JSON → StockRegimeAssessment
    as_json = json.dumps(report_dict)
    restored = StockRegimeAssessment.model_validate_json(as_json)
    print(f"Round-trip OK : regime={restored.regime}, evaluation={restored.evaluation}")

## 8 — Agent tools layer

`tools.py` wraps `chains.py` (and other pipeline functions) as LangChain `@tool`
functions so a LangGraph agent can call them autonomously.

### Tool inventory

| Tool function | Calls | Returns |
|---|---|---|
| `list_available_tickers()` | `list_evaluated_tickers(base_dir)` | JSON array of ticker strings |
| `get_stock_metrics(ticker, year?)` | `get_fin_data(base_dir)` | JSON `{metrics, composite_scores, red_flags}` |
| `get_sector_benchmarks(sector)` | `get_market_metrics(base_dir)` × 2 | JSON `{financial, valuation}` |
| `get_red_flags(ticker)` | `get_fin_data(base_dir)` | JSON `{red_flags}` |
| `get_stock_regime_report(ticker, year?)` | `get_stock_evaluation_report(base_dir, sector_file)` | JSON of `StockRegimeAssessment.model_dump()` |

### Design invariants

- All tools return **JSON strings** — never Python objects, never `None`.
- Errors arrive as `{"error": "..."}` — the agent can reason about them without a traceback.
- **File paths are deployment config, not agent decisions.** The LLM never sees `base_dir`
  or `sector_file` as tool parameters. They are baked in at bootstrap time via `make_tools()`.

### Bootstrap patterns

```python
from financialtools.tools import make_tools, TOOLS

# In-repo default — financial_data/ relative to CWD:
agent = create_agent(model=llm, tools=TOOLS, ...)

# External consumer — explicit absolute paths:
tools = make_tools(
    base_dir="/path/to/my/financial_data",
    sector_file="/path/to/my/sector_ticker.txt",
)
agent = create_agent(model=llm, tools=tools, ...)
```

In [ ]:
from financialtools.tools import make_tools, TOOLS

# Inspect the default tool registry (offline — no API call)
print(f"Number of tools : {len(TOOLS)}")
print()
for t in TOOLS:
    print(f"  {t.name:<30} {t.description.splitlines()[0][:65]}")

print()

# make_tools() produces the same tool names/descriptions regardless of paths —
# the LLM sees identical tool signatures; only the baked-in paths differ.
custom_tools = make_tools(base_dir="/tmp/data", sector_file="/tmp/sectors.txt")
print(f"custom_tools names match TOOLS: {[t.name for t in custom_tools] == [t.name for t in TOOLS]}")

In [ ]:
from financialtools.tools import list_available_tickers, get_stock_regime_report

# list_available_tickers — works without API key; returns [] if no data files exist
raw = list_available_tickers.invoke({})
tickers = json.loads(raw)
print(f"list_available_tickers: {tickers if tickers else '[] (no financial_data dir yet)'}")

# Demonstrate error envelope: tool never raises
if not os.environ.get("OPENAI_API_KEY"):
    raw_report = get_stock_regime_report.invoke({"ticker": "AAPL"})
    result_obj = json.loads(raw_report)
    if "error" in result_obj:
        print(f"\nget_stock_regime_report returns error envelope (no key):")
        print(f"  {result_obj}")
    else:
        print(f"\nget_stock_regime_report: regime={result_obj.get('regime')}")

## 9 — Error handling reference

| Error | Raised by | Condition | Caught by |
|---|---|---|---|
| `EnvironmentError` | caller | `OPENAI_API_KEY` not set | application code |
| `FileNotFoundError` | `read_financial_results` | `financial_data/*.xlsx` missing | `get_stock_metrics`, `get_red_flags`, `get_stock_regime_report` — returns `{"error": ...}` |
| `SectorNotFoundError` | `get_sector_for_ticker` | ticker absent from `sector_ticker.txt` | `get_stock_regime_report` — returns `{"error": ...}` |
| `SectorNotFoundError` | `get_market_metrics` | sector absent from benchmark Excel | `get_sector_benchmarks` — returns `{"error": ...}` |
| `OutputParserException` | `PydanticOutputParser` | LLM returned invalid JSON or bad `Literal` | `OutputFixingParser` makes a second LLM call |
| Any exception from the second retry | `OutputFixingParser` | Both parse attempts fail | propagates up; caught by `get_stock_regime_report` → `{"error": ...}` |

### Debugging path

1. **Tool returns `{"error": "..."}`** — read the message; it names the missing file or
   missing ticker. Resolve the prerequisite, then retry.
2. **`OutputFixingParser` raises** — the LLM produced non-conforming output twice.
   Check `logs/error.log`; consider switching to a more capable model or adding
   `{format_instructions}` to the prompt.
3. **`SectorNotFoundError` for a known ticker** — the ticker is not in
   `financialtools/data/sector_ticker.txt`. Add `<TICKER>\t<sector>\t<name>\t1`.

In [ ]:
from financialtools.exceptions import SectorNotFoundError
from langchain_core.exceptions import OutputParserException

# 1. Bad Literal value → OutputParserException
bad_literal = json.dumps({
    "ticker": "AAPL", "regime": "sideways",   # invalid — must be bull or bear
    "regime_rationale": "...", "metrics_movement": "...",
    "non_aligned_findings": None,
    "evaluation": "fair", "evaluation_rationale": "...", "market_comparison": "...",
})
try:
    base_parser.parse(bad_literal)
except OutputParserException as exc:
    print(f"OutputParserException (bad Literal): {str(exc)[:100]}")

# 2. Missing required field → OutputParserException
missing_field = json.dumps({"ticker": "AAPL", "regime": "bull"})
try:
    base_parser.parse(missing_field)
except OutputParserException as exc:
    print(f"OutputParserException (missing fields): {str(exc)[:100]}")

# 3. SectorNotFoundError for unknown ticker (offline — no Excel read required)
from financialtools.utils import get_sector_for_ticker
try:
    get_sector_for_ticker("NONEXISTENT_TICKER_XYZ")
except SectorNotFoundError as exc:
    print(f"SectorNotFoundError: {exc}")
except FileNotFoundError as exc:
    print(f"FileNotFoundError (sector_ticker.txt missing): {exc}")

## 10 — End-to-end pattern

Full pipeline from ticker symbol to structured regime report, with a guard pattern
that checks prerequisites before making any API call.

> **LIVE cell** — requires `OPENAI_API_KEY` and all `financial_data/*.xlsx` files.

In [ ]:
from financialtools.tools import list_available_tickers, get_stock_regime_report
from financialtools.exceptions import SectorNotFoundError

_has_key  = bool(os.environ.get("OPENAI_API_KEY"))
_tickers  = json.loads(list_available_tickers.invoke({}))
_has_data = len(_tickers) > 0

if not _has_key:
    print("Skipping — OPENAI_API_KEY not set.")
elif not _has_data:
    print("Skipping — no evaluated tickers found. Run: python scripts/run_pipeline.py")
else:
    TARGET_TICKER = _tickers[0]      # pick the first available ticker
    TARGET_YEAR   = 2023             # set to None for all years

    print(f"Available tickers : {_tickers}")
    print(f"Analysing         : {TARGET_TICKER}  year={TARGET_YEAR}")
    print()

    # --- Call via tool (agent-compatible) ---
    raw = get_stock_regime_report.invoke({"ticker": TARGET_TICKER, "year": TARGET_YEAR})
    result = json.loads(raw)

    if "error" in result:
        print(f"Tool returned error: {result['error']}")
    else:
        print(f"Regime       : {result['regime']}")
        print(f"Evaluation   : {result['evaluation']}")
        print()
        print("Regime rationale:")
        print(f"  {result['regime_rationale']}")
        print()
        print("Market comparison:")
        print(f"  {result['market_comparison']}")